# 10 — Statistical analysis and manuscript figures

This notebook consumes the final benchmark outputs generated by `09_final_model_family_benchmark_public.ipynb`.

It performs the final statistical comparison across model families and generates the manuscript figures. It does not fit, tune, or select forecasting models.

The analysis includes:

- Friedman comparison across model families;
- pairwise Wilcoxon signed-rank contrasts with Holm correction;
- paired bootstrap confidence intervals for aggregated NRMSE differences;
- task-level Diebold–Mariano tests using a Newey–West long-run variance estimate;
- manuscript-ready Figures 3, 4, and 5 with their corresponding source-data CSV files.

## Interpretation

NRMSE is used for cross-task comparison because temperature and relative humidity have different units. RMSE and $R^2$ are retained for within-target descriptive reporting.

The main contrast is **Robust hybrid versus Advanced traditional**. Statistical non-rejection is not interpreted as equivalence or superiority. Results are interpreted as task-dependent performance.

In [ ]:
from __future__ import annotations

import itertools
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
def locate_repository_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "results" / "final_benchmark").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the repository."
    )


PROJECT_ROOT = locate_repository_root()

BENCHMARK_DIR = PROJECT_ROOT / "results" / "final_benchmark"
RESULTS_DIR = PROJECT_ROOT / "results" / "statistical_analysis"
FIGURE_DIR = PROJECT_ROOT / "figures" / "article"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TASK_COLUMNS = ["resolution_minutes", "target", "horizon_minutes"]
KEY_COLUMNS = TASK_COLUMNS + ["origin_index"]

FAMILIES = [
    "BASELINE",
    "STATISTICAL",
    "MACHINE_LEARNING",
    "DEEP_LEARNING",
    "ADVANCED_TRADITIONAL",
    "HYBRID_ROBUST",
]

FAMILY_LABELS = {
    "BASELINE": "Operational reference",
    "STATISTICAL": "Statistical",
    "MACHINE_LEARNING": "Machine learning",
    "DEEP_LEARNING": "Deep learning",
    "ADVANCED_TRADITIONAL": "Advanced traditional",
    "HYBRID_ROBUST": "Robust hybrid",
}

FAMILY_ABBREVIATIONS = {
    "BASELINE": "B",
    "STATISTICAL": "S",
    "MACHINE_LEARNING": "ML",
    "DEEP_LEARNING": "DL",
    "ADVANCED_TRADITIONAL": "AT",
    "HYBRID_ROBUST": "RH",
}

FAMILY_COLORS = {
    "BASELINE": "#9E9E9E",
    "STATISTICAL": "#4C78A8",
    "MACHINE_LEARNING": "#F2CF5B",
    "DEEP_LEARNING": "#B279A2",
    "ADVANCED_TRADITIONAL": "#59A14F",
    "HYBRID_ROBUST": "#E15759",
    "SHARED": "#D9D9D9",
}

NUMERICAL_TIE_TOLERANCE = 1e-6
ALPHA = 0.05
BOOTSTRAP_REPETITIONS = 5000
RANDOM_SEED = 2026

SOURCE_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "greenhouse_sensor_data_4min.csv"
SPLIT_BOUNDARIES_FILE = (
    PROJECT_ROOT / "results" / "multiresolution" / "05_chronological_split_boundaries.csv"
)

print("Repository structure located successfully.")
print("Statistical analysis configuration ready.")

## Load final benchmark outputs

In [ ]:
TASK_METRICS_FILE = BENCHMARK_DIR / "05_final_family_task_metrics.csv"
PREDICTIONS_FILE = ( BENCHMARK_DIR / "predictions" / "01_family_selected_test_predictions.parquet")

required_inputs = [ TASK_METRICS_FILE, PREDICTIONS_FILE, SOURCE_DATA_FILE, SPLIT_BOUNDARIES_FILE,]

for path in required_inputs:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path.name}")

task_metrics = pd.read_csv(TASK_METRICS_FILE)
predictions = pd.read_parquet(PREDICTIONS_FILE)

required_metric_columns = { "family", *TASK_COLUMNS, "rmse", "r2", "nrmse", "mae", "bias",}
required_prediction_columns = { "family", *KEY_COLUMNS, "observed", "predicted",}

missing_metric_columns = sorted(required_metric_columns - set(task_metrics.columns))
missing_prediction_columns = sorted(required_prediction_columns - set(predictions.columns))

if missing_metric_columns:
    raise ValueError(f"Task metrics missing columns: {missing_metric_columns}")
if missing_prediction_columns:
    raise ValueError(f"Predictions missing columns: {missing_prediction_columns}")

metric_families = sorted(task_metrics["family"].dropna().unique())
prediction_families = sorted(predictions["family"].dropna().unique())

if set(metric_families) != set(FAMILIES):
    raise ValueError(f"Unexpected model families in task metrics: {metric_families}")
if set(prediction_families) != set(FAMILIES):
    raise ValueError(f"Unexpected model families in predictions: {prediction_families}")

if task_metrics.duplicated(["family", *TASK_COLUMNS]).any():
    raise ValueError("Duplicate family-task rows were found in the benchmark metrics.")

task_counts = (
    task_metrics.groupby("family", observed=True)
    .size()
    .rename("tasks")
    .reset_index()
)

if task_counts["tasks"].nunique() != 1:
    raise ValueError("Model families do not contain the same set of forecasting tasks.")

input_summary = pd.DataFrame([
    {
        "input": "results/final_benchmark/05_final_family_task_metrics.csv",
        "rows": len(task_metrics),
        "columns": len(task_metrics.columns),
    },
    {
        "input": "results/final_benchmark/predictions/01_family_selected_test_predictions.parquet",
        "rows": len(predictions),
        "columns": len(predictions.columns),
    },
])

input_summary.to_csv(RESULTS_DIR / "01_input_summary.csv", index=False)

display(task_counts)
display(input_summary)

## Figure 3 — Temperature and relative humidity time series

The validated 4-minute SHT31 series is aggregated to hourly resolution for visualization only. Chronological training, validation, and test boundaries are read from notebook 03 outputs.

In [ ]:
split_boundaries = pd.read_csv(SPLIT_BOUNDARIES_FILE)
split_boundaries["start_inclusive"] = pd.to_datetime(
    split_boundaries["start_inclusive"], errors="raise"
)
split_boundaries["end_exclusive"] = pd.to_datetime(
    split_boundaries["end_exclusive"], errors="raise"
)
split_boundaries = split_boundaries.set_index("split")

required_splits = {"train", "validation", "test"}
if set(split_boundaries.index) != required_splits:
    raise ValueError(
        f"Chronological split table must contain train, validation and test: "
        f"{list(split_boundaries.index)}"
    )

TRAIN_START = split_boundaries.loc["train", "start_inclusive"]
VALIDATION_START = split_boundaries.loc["validation", "start_inclusive"]
TEST_START = split_boundaries.loc["test", "start_inclusive"]
STUDY_END = split_boundaries.loc["test", "end_exclusive"]

available_columns = pd.read_csv(
    SOURCE_DATA_FILE, nrows=0, encoding="utf-8-sig"
).columns.tolist()

target_column_candidates = {
    "temperature": ["temp_sht31_clean", "temperature", "temp_sht31"],
    "relative_humidity": ["rh_sht31_clean", "relative_humidity", "rh_sht31"],
}

selected_target_columns = {}
for target, candidates in target_column_candidates.items():
    selected = next(
        (column for column in candidates if column in available_columns),
        None,
    )
    if selected is None:
        raise ValueError(
            f"No compatible source column was found for {target}. "
            f"Available columns: {available_columns}"
        )
    selected_target_columns[target] = selected

if "timestamp" not in available_columns:
    raise ValueError("The 4-minute dataset does not contain a timestamp column.")

micro = pd.read_csv(
    SOURCE_DATA_FILE,
    usecols=["timestamp", *selected_target_columns.values()],
    encoding="utf-8-sig",
)

micro["timestamp"] = pd.to_datetime(micro["timestamp"], errors="coerce")

if micro["timestamp"].isna().any():
    raise ValueError("The 4-minute series contains invalid timestamps.")
if micro["timestamp"].duplicated().any():
    raise ValueError("The 4-minute series contains duplicate timestamps.")

micro = (
    micro.rename(
        columns={
            column: target
            for target, column in selected_target_columns.items()
        }
    )
    .sort_values("timestamp")
    .set_index("timestamp")
)

micro_hourly = (
    micro[["temperature", "relative_humidity"]]
    .resample("1h")
    .mean()
)
micro_hourly = micro_hourly.loc[
    (micro_hourly.index >= TRAIN_START)
    & (micro_hourly.index < STUDY_END)
]

figure_03_data = micro_hourly.reset_index()
figure_03_data.to_csv(RESULTS_DIR / "figure_03_data.csv", index=False)

fig, (ax_temperature, ax_humidity) = plt.subplots(
    2, 1, figsize=(12.5, 6.8), sharex=True
)

ax_temperature.plot(
    micro_hourly.index,
    micro_hourly["temperature"],
    linewidth=0.75,
    color="#1f77b4",
)
ax_humidity.plot(
    micro_hourly.index,
    micro_hourly["relative_humidity"],
    linewidth=0.75,
    color="#1f77b4",
)

for axis in (ax_temperature, ax_humidity):
    axis.axvline(
        VALIDATION_START,
        color="black",
        linestyle="--",
        linewidth=1.1,
    )
    axis.axvline(
        TEST_START,
        color="black",
        linestyle="--",
        linewidth=1.1,
    )
    axis.grid(alpha=0.20, linewidth=0.6)
    axis.set_xlim(TRAIN_START, STUDY_END)

ax_temperature.text(
    0.012, 0.93, "(a)",
    transform=ax_temperature.transAxes,
    va="top",
    ha="left",
    fontweight="semibold",
)
ax_humidity.text(
    0.012, 0.93, "(b)",
    transform=ax_humidity.transAxes,
    va="top",
    ha="left",
    fontweight="semibold",
)

partition_centers = {
    "Training": TRAIN_START + (VALIDATION_START - TRAIN_START) / 2,
    "Validation": VALIDATION_START + (TEST_START - VALIDATION_START) / 2,
    "Test": TEST_START + (STUDY_END - TEST_START) / 2,
}

for label, position in partition_centers.items():
    ax_temperature.text(
        position,
        1.025,
        label,
        transform=ax_temperature.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=11,
        clip_on=False,
    )

ax_temperature.set_ylabel("Air temperature [°C]")
ax_humidity.set_ylabel("Relative humidity [%]")
ax_humidity.set_xlabel("Date")

ax_humidity.xaxis.set_major_locator(mdates.MonthLocator())
ax_humidity.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

for boundary in [VALIDATION_START, TEST_START]:
    ax_humidity.annotate(
        boundary.strftime("%b %d"),
        xy=(boundary, 0),
        xycoords=("data", "axes fraction"),
        xytext=(0, -26),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=10.5,
    )

fig.subplots_adjust(
    left=0.10,
    right=0.985,
    bottom=0.15,
    top=0.91,
    hspace=0.12,
)

for extension in ["png", "pdf"]:
    fig.savefig(
        FIGURE_DIR / f"03_temporal_microclimate.{extension}",
        dpi=600,
        bbox_inches="tight",
    )

plt.show()

## Friedman test across paired forecasting tasks

In [ ]:
nrmse_panel = task_metrics.pivot(index=TASK_COLUMNS, columns="family", values="nrmse")
nrmse_panel = nrmse_panel.reindex(columns=sorted(FAMILIES))
assert nrmse_panel.notna().all().all()

friedman_statistic, friedman_p_value = stats.friedmanchisquare(
    *[nrmse_panel[family].to_numpy(dtype=float) for family in nrmse_panel.columns]
)
friedman_result = pd.DataFrame([{
    "n_complete_tasks": len(nrmse_panel),
    "n_families": len(nrmse_panel.columns),
    "families": ";".join(nrmse_panel.columns),
    "metric": "NRMSE normalized by test-task target SD",
    "friedman_statistic": friedman_statistic,
    "p_value": friedman_p_value,
    "significant": friedman_p_value < ALPHA,
}])
friedman_result.to_csv(RESULTS_DIR / "02_friedman_family_comparison.csv", index=False)
display(friedman_result)

## Pairwise Wilcoxon signed-rank tests with Holm correction

In [ ]:
def paired_wilcoxon(family_a, family_b):
    difference = (
        nrmse_panel[family_a].to_numpy(dtype=float)
        - nrmse_panel[family_b].to_numpy(dtype=float)
    )
    numerical_ties = np.abs(difference) <= NUMERICAL_TIE_TOLERANCE
    adjusted_difference = difference.copy()
    adjusted_difference[numerical_ties] = 0.0
    nonzero = adjusted_difference[~numerical_ties]
    if len(nonzero) == 0:
        statistic, p_value = 0.0, 1.0
    else:
        result = stats.wilcoxon(nonzero, zero_method="wilcox", alternative="two-sided", method="auto")
        statistic, p_value = float(result.statistic), float(result.pvalue)
    favoring_a = int((adjusted_difference < 0).sum())
    favoring_b = int((adjusted_difference > 0).sum())
    n_nonzero = favoring_a + favoring_b
    rank_biserial_favoring_a = (favoring_a - favoring_b) / n_nonzero if n_nonzero else 0.0
    mean_difference = float(np.mean(adjusted_difference))
    if abs(mean_difference) <= NUMERICAL_TIE_TOLERANCE:
        direction = "NO_MEAN_DIFFERENCE"
    elif mean_difference < 0:
        direction = f"{family_a}_better"
    else:
        direction = f"{family_b}_better"
    return {
        "family_a": family_a,
        "family_b": family_b,
        "n_tasks": len(difference),
        "n_nonzero_tasks": n_nonzero,
        "n_numerical_ties": int(numerical_ties.sum()),
        "tasks_favoring_a": favoring_a,
        "tasks_favoring_b": favoring_b,
        "numerical_tie_tolerance": NUMERICAL_TIE_TOLERANCE,
        "median_nrmse_difference_a_minus_b": float(np.median(adjusted_difference)),
        "mean_nrmse_difference_a_minus_b": mean_difference,
        "rank_biserial_favoring_a": rank_biserial_favoring_a,
        "wilcoxon_statistic": statistic,
        "p_value": p_value,
        "direction": direction,
    }


wilcoxon_rows = [
    paired_wilcoxon(a, b)
    for a, b in itertools.combinations(sorted(FAMILIES), 2)
]
wilcoxon_results = pd.DataFrame(wilcoxon_rows)
wilcoxon_results["p_value_holm"] = multipletests(
    wilcoxon_results["p_value"], alpha=ALPHA, method="holm"
)[1]
wilcoxon_results["significant_after_holm"] = wilcoxon_results["p_value_holm"] < ALPHA
wilcoxon_results.to_csv(RESULTS_DIR / "03_pairwise_family_wilcoxon_holm.csv", index=False)
display(wilcoxon_results)

## Paired task bootstrap

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
for family_a, family_b in itertools.combinations(sorted(FAMILIES), 2):
    difference = (
        nrmse_panel[family_a].to_numpy(dtype=float)
        - nrmse_panel[family_b].to_numpy(dtype=float)
    )
    difference[np.abs(difference) <= NUMERICAL_TIE_TOLERANCE] = 0.0
    sample_indices = rng.integers(0, len(difference), size=(BOOTSTRAP_REPETITIONS, len(difference)))
    samples = difference[sample_indices]
    boot_mean = samples.mean(axis=1)
    boot_median = np.median(samples, axis=1)
    mean_ci = np.quantile(boot_mean, [0.025, 0.975])
    median_ci = np.quantile(boot_median, [0.025, 0.975])
    bootstrap_rows.append({
        "family_a": family_a, "family_b": family_b,
        "metric": "NRMSE normalized by test-task target SD",
        "n_tasks": len(difference), "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "mean_difference": float(difference.mean()),
        "median_difference": float(np.median(difference)),
        "mean_ci_lower": float(mean_ci[0]), "mean_ci_upper": float(mean_ci[1]),
        "median_ci_lower": float(median_ci[0]), "median_ci_upper": float(median_ci[1]),
        "mean_ci_excludes_zero": bool(mean_ci[0] > 0 or mean_ci[1] < 0),
        "median_ci_excludes_zero": bool(median_ci[0] > 0 or median_ci[1] < 0),
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_results.to_csv(RESULTS_DIR / "04_family_paired_bootstrap.csv", index=False)
display(bootstrap_results)

## Diebold–Mariano tests with Newey–West variance

For each task and each contrast involving the robust hybrid strategy, the loss differential is squared error of `HYBRID_ROBUST` minus squared error of the comparison family. A negative mean differential favors the hybrid strategy. The Bartlett-weighted Newey–West lag is the forecast horizon in resolution steps minus one, capped at $n-1$. Holm correction is applied globally across all finite task-level tests.

In [ ]:
def newey_west_long_run_variance(values, maximum_lag):
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    n = len(centered)
    maximum_lag = int(min(maximum_lag, n - 1))
    variance = float(np.dot(centered, centered) / n)
    for lag in range(1, maximum_lag + 1):
        weight = 1.0 - lag / (maximum_lag + 1.0)
        covariance = float(np.dot(centered[lag:], centered[:-lag]) / n)
        variance += 2.0 * weight * covariance
    return max(variance, 0.0)


def dm_test(loss_differential, maximum_lag):
    difference = np.asarray(loss_differential, dtype=float)
    if np.all(np.abs(difference) <= 1e-14):
        return 0.0, 1.0, 0.0, 0.0, "exact_loss_tie"
    long_run_variance = newey_west_long_run_variance(difference, maximum_lag)
    standard_error = np.sqrt(long_run_variance / len(difference))
    if not np.isfinite(standard_error) or standard_error <= 0:
        return np.nan, np.nan, float(difference.mean()), long_run_variance, "nonpositive_long_run_variance"
    statistic = float(difference.mean() / standard_error)
    p_value = float(2.0 * stats.norm.sf(abs(statistic)))
    return statistic, p_value, float(difference.mean()), long_run_variance, "ok"


wide_predictions = predictions.pivot(
    index=KEY_COLUMNS + ["observed"], columns="family", values="predicted"
).reset_index()
dm_rows = []
for comparison_family in [family for family in FAMILIES if family != "HYBRID_ROBUST"]:
    for task_keys, group in wide_predictions.groupby(TASK_COLUMNS, sort=False, observed=True):
        observed = group["observed"].to_numpy(dtype=float)
        hybrid_error = group["HYBRID_ROBUST"].to_numpy(dtype=float) - observed
        comparison_error = group[comparison_family].to_numpy(dtype=float) - observed
        loss_difference = hybrid_error ** 2 - comparison_error ** 2
        resolution, target, horizon = task_keys
        horizon_steps = max(1, int(round(horizon / resolution)))
        maximum_lag = min(horizon_steps - 1, len(group) - 1)
        statistic, p_value, mean_difference, long_run_variance, status = dm_test(
            loss_difference, maximum_lag
        )
        dm_rows.append({
            "family_a": "HYBRID_ROBUST", "family_b": comparison_family,
            "resolution_minutes": int(resolution), "target": target,
            "horizon_minutes": int(horizon), "n_origins": len(group),
            "newey_west_lag": maximum_lag,
            "mean_squared_error_difference_a_minus_b": mean_difference,
            "newey_west_long_run_variance": long_run_variance,
            "dm_statistic": statistic, "p_value": p_value, "status": status,
            "direction": (
                "HYBRID_ROBUST_better" if mean_difference < -1e-14
                else f"{comparison_family}_better" if mean_difference > 1e-14
                else "EXACT_TIE"
            ),
        })

dm_results = pd.DataFrame(dm_rows)
finite = dm_results["p_value"].notna()
dm_results["p_value_holm_global"] = np.nan
if finite.any():
    dm_results.loc[finite, "p_value_holm_global"] = multipletests(
        dm_results.loc[finite, "p_value"], alpha=ALPHA, method="holm"
    )[1]
dm_results["significant_after_global_holm"] = dm_results["p_value_holm_global"] < ALPHA
dm_results.to_csv(RESULTS_DIR / "05_dm_hybrid_primary_contrasts.csv", index=False)

dm_summary = (
    dm_results.groupby(["family_a", "family_b"], as_index=False)
    .agg(
        tasks=("p_value", "size"), exact_ties=("status", lambda s: int((s == "exact_loss_tie").sum())),
        hybrid_better_tasks=("direction", lambda s: int((s == "HYBRID_ROBUST_better").sum())),
        comparison_better_tasks=("direction", lambda s: int(s.str.endswith("_better").sum() - (s == "HYBRID_ROBUST_better").sum())),
        significant_tasks_global_holm=("significant_after_global_holm", "sum"),
    )
)
dm_summary.to_csv(RESULTS_DIR / "06_dm_hybrid_primary_summary.csv", index=False)
display(dm_summary)

## Descriptive task winners and the primary hybrid contrast

In [ ]:
winner_rows = []
for task_keys, group in task_metrics.groupby(TASK_COLUMNS, sort=False, observed=True):
    minimum_rmse = float(group["rmse"].min())
    winners = sorted(group.loc[np.abs(group["rmse"] - minimum_rmse) <= NUMERICAL_TIE_TOLERANCE, "family"])
    winner_rows.append({
        "resolution_minutes": int(task_keys[0]), "target": task_keys[1],
        "horizon_minutes": int(task_keys[2]), "minimum_rmse": minimum_rmse,
        "winning_families": ";".join(winners), "n_winners": len(winners),
        "winner_label": " / ".join(FAMILY_ABBREVIATIONS[x] for x in winners),
        "winner_names": " / ".join(FAMILY_LABELS[x] for x in winners),
        "winner_category": winners[0] if len(winners) == 1 else "SHARED",
    })
task_winners = pd.DataFrame(winner_rows)
task_winners.to_csv(RESULTS_DIR / "07_task_winners_rmse.csv", index=False)
task_winners.to_csv(RESULTS_DIR / "figure_04_data.csv", index=False)

hybrid_comparison = task_metrics.loc[
    task_metrics["family"].isin(["ADVANCED_TRADITIONAL", "HYBRID_ROBUST"])
].pivot(index=TASK_COLUMNS, columns="family", values="rmse").reset_index()
hybrid_comparison["hybrid_gain_pct"] = 100.0 * (
    hybrid_comparison["ADVANCED_TRADITIONAL"] - hybrid_comparison["HYBRID_ROBUST"]
) / hybrid_comparison["ADVANCED_TRADITIONAL"]
hybrid_comparison["outcome"] = np.select(
    [
        np.abs(hybrid_comparison["HYBRID_ROBUST"] - hybrid_comparison["ADVANCED_TRADITIONAL"]) <= NUMERICAL_TIE_TOLERANCE,
        hybrid_comparison["HYBRID_ROBUST"] < hybrid_comparison["ADVANCED_TRADITIONAL"],
    ],
    ["NUMERICAL_TIE", "HYBRID_ROBUST_BETTER"],
    default="ADVANCED_TRADITIONAL_BETTER",
)
hybrid_comparison.loc[hybrid_comparison["outcome"].eq("NUMERICAL_TIE"), "hybrid_gain_pct"] = 0.0
hybrid_comparison.to_csv(RESULTS_DIR / "08_hybrid_vs_advanced_task_comparison.csv", index=False)
hybrid_comparison.to_csv(RESULTS_DIR / "figure_05_data.csv", index=False)
hybrid_summary = hybrid_comparison["outcome"].value_counts().rename_axis("outcome").reset_index(name="tasks")
hybrid_summary.to_csv(RESULTS_DIR / "09_hybrid_vs_advanced_summary.csv", index=False)
display(hybrid_summary)

## Figure 4 — Task-winning model family

In [ ]:
target_order = ["temperature", "relative_humidity"]
target_labels = {"temperature": "Temperature", "relative_humidity": "Relative humidity"}
target_units = {"temperature": "°C", "relative_humidity": "p.p."}
resolutions = sorted(task_winners["resolution_minutes"].unique())
horizons = sorted(task_winners["horizon_minutes"].unique())

family_cell_labels = {
    "Operational reference": "Operational\nreference",
    "Statistical": "Statistical",
    "Machine learning": "Machine\nlearning",
    "Deep learning": "Deep\nlearning",
    "Advanced traditional": "Advanced\ntraditional",
    "Robust hybrid": "Robust\nhybrid",
}

def winner_text_for_cell(names):
    return "\n/\n".join(family_cell_labels.get(name.strip(), name.strip()) for name in names.split(" / "))

def contrast_text_color(image, value):
    red, green, blue, _ = image.cmap(image.norm(value))
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return "black" if luminance > 0.58 else "white"

fig = plt.figure(figsize=(18.5, 7.2))
grid = fig.add_gridspec(1, 4, width_ratios=[1, 0.045, 1, 0.045], wspace=0.26)
axes = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 2])]
colorbar_axes = [fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[0, 3])]
axes[1].sharey(axes[0])

for panel_index, (axis, colorbar_axis, target) in enumerate(zip(axes, colorbar_axes, target_order)):
    subset = task_winners.loc[task_winners["target"].eq(target)]
    rmse_matrix = subset.pivot(index="resolution_minutes", columns="horizon_minutes", values="minimum_rmse").reindex(index=resolutions, columns=horizons)
    image = axis.imshow(rmse_matrix.to_numpy(dtype=float), aspect="auto", cmap="viridis")
    for row_index, resolution in enumerate(resolutions):
        for column_index, horizon in enumerate(horizons):
            cell = subset.loc[
                subset["resolution_minutes"].eq(resolution)
                & subset["horizon_minutes"].eq(horizon)
            ].iloc[0]
            rmse = float(cell["minimum_rmse"])
            label = winner_text_for_cell(cell["winner_names"])
            axis.text(
                column_index, row_index, f"{label}\nRMSE = {rmse:.2f}",
                ha="center", va="center", fontsize=9.2, fontweight="semibold",
                linespacing=1.02, color=contrast_text_color(image, rmse),
            )
    axis.set_xticks(range(len(horizons)), [str(value) for value in horizons], fontsize=12)
    axis.set_yticks(range(len(resolutions)), [str(value) for value in resolutions], fontsize=12)
    axis.set_xlabel("Forecast horizon [min]", fontsize=14)
    axis.set_title(f"({'ab'[panel_index]}) {target_labels[target]}", fontsize=16)
    axis.set_xticks(np.arange(-0.5, len(horizons), 1), minor=True)
    axis.set_yticks(np.arange(-0.5, len(resolutions), 1), minor=True)
    axis.grid(which="minor", color="white", linewidth=0.8, alpha=0.45)
    axis.tick_params(which="minor", bottom=False, left=False)
    colorbar = fig.colorbar(image, cax=colorbar_axis)
    colorbar.set_label(f"Minimum RMSE [{target_units[target]}]", fontsize=13, labelpad=8)
    colorbar.ax.tick_params(labelsize=11)

axes[0].set_ylabel("Temporal resolution [min]", fontsize=14)
axes[1].tick_params(axis="y", which="both", left=False, labelleft=False)
axes[1].spines["left"].set_visible(False)
fig.subplots_adjust(left=0.07, right=0.96, bottom=0.14, top=0.90)
for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"04_family_winners_rmse.{extension}", dpi=600, bbox_inches="tight")
plt.show()

## Figure 5 — Robust hybrid RMSE gain relative to Advanced traditional

In [ ]:
gain_limit = max(1.0, float(np.nanmax(np.abs(hybrid_comparison["hybrid_gain_pct"]))))
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), constrained_layout=True, sharey=True)
for panel_index, (axis, target) in enumerate(zip(axes, target_order)):
    matrix = hybrid_comparison.loc[hybrid_comparison["target"].eq(target)].pivot(
        index="resolution_minutes", columns="horizon_minutes", values="hybrid_gain_pct"
    ).reindex(index=resolutions, columns=horizons)
    sns.heatmap(
        matrix, ax=axis, cmap="RdBu_r", center=0, vmin=-gain_limit, vmax=gain_limit,
        annot=True, fmt=".2f", linewidths=0.75, linecolor="white",
        cbar=axis is axes[-1],
        cbar_kws={"label": "Relative change in RMSE [%]\npositive values favor the hybrid"} if axis is axes[-1] else None,
    )
    axis.set_xlabel("Forecast horizon [min]", fontsize=12)
    axis.set_ylabel("Temporal resolution [min]" if panel_index == 0 else "", fontsize=12)
    axis.set_title(f"({'ab'[panel_index]}) {target_labels[target]}", fontsize=14)
    axis.set_yticklabels(axis.get_yticklabels(), rotation=0)

for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"05_hybrid_vs_advanced_rmse_gain.{extension}", dpi=600, bbox_inches="tight")
plt.show()

## Supplementary average-rank figure

In [ ]:
average_ranks = (
    task_metrics.assign(
        rmse_rank=lambda x: x.groupby(TASK_COLUMNS)["rmse"].rank(method="average")
    ).groupby("family", as_index=False)
    .agg(average_rmse_rank=("rmse_rank", "mean"), median_rmse_rank=("rmse_rank", "median"))
    .sort_values("average_rmse_rank")
)
average_ranks["family_label"] = average_ranks["family"].map(FAMILY_LABELS)
average_ranks.to_csv(RESULTS_DIR / "10_average_family_ranks.csv", index=False)

fig, axis = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
axis.barh(
    average_ranks["family_label"], average_ranks["average_rmse_rank"],
    color=[FAMILY_COLORS[x] for x in average_ranks["family"]],
)
axis.invert_yaxis()
axis.set_xlabel("Average RMSE rank (lower is better)")
axis.set_ylabel("")
axis.set_title("Average family rank across paired forecasting tasks", fontweight="bold")
axis.set_xlim(0.5, len(FAMILIES) + 0.2)
for row, value in enumerate(average_ranks["average_rmse_rank"]):
    axis.text(value + 0.05, row, f"{value:.2f}", va="center")
for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"S01_average_family_rank.{extension}", dpi=300, bbox_inches="tight")
plt.show()

## Manuscript-ready statistical summary

In [ ]:
primary_wilcoxon = wilcoxon_results.loc[
    (
        wilcoxon_results["family_a"].eq("ADVANCED_TRADITIONAL")
        & wilcoxon_results["family_b"].eq("HYBRID_ROBUST")
    )
    | (
        wilcoxon_results["family_a"].eq("HYBRID_ROBUST")
        & wilcoxon_results["family_b"].eq("ADVANCED_TRADITIONAL")
    )
].iloc[0]

primary_bootstrap = bootstrap_results.loc[
    (
        bootstrap_results["family_a"].eq("ADVANCED_TRADITIONAL")
        & bootstrap_results["family_b"].eq("HYBRID_ROBUST")
    )
    | (
        bootstrap_results["family_a"].eq("HYBRID_ROBUST")
        & bootstrap_results["family_b"].eq("ADVANCED_TRADITIONAL")
    )
].iloc[0]

if primary_wilcoxon["p_value_holm"] < ALPHA:
    primary_interpretation = (
        "The Holm-adjusted paired comparison detected an overall difference "
        "between Advanced traditional and Robust hybrid."
    )
else:
    primary_interpretation = (
        "The Holm-adjusted paired comparison did not detect an overall difference "
        "between Advanced traditional and Robust hybrid."
    )

statistical_summary = pd.DataFrame([
    {
        "analysis": "Friedman omnibus test",
        "statistic": float(friedman_statistic),
        "p_value": float(friedman_p_value),
        "ci_lower": np.nan,
        "ci_upper": np.nan,
        "interpretation": (
            "Model-family performance differs across the paired forecasting tasks."
            if friedman_p_value < ALPHA
            else "No overall model-family difference was detected."
        ),
    },
    {
        "analysis": "Advanced traditional vs Robust hybrid — Wilcoxon with Holm correction",
        "statistic": float(primary_wilcoxon["wilcoxon_statistic"]),
        "p_value": float(primary_wilcoxon["p_value_holm"]),
        "ci_lower": np.nan,
        "ci_upper": np.nan,
        "interpretation": primary_interpretation,
    },
    {
        "analysis": "Advanced traditional minus Robust hybrid — paired bootstrap mean NRMSE",
        "statistic": float(primary_bootstrap["mean_difference"]),
        "p_value": np.nan,
        "ci_lower": float(primary_bootstrap["mean_ci_lower"]),
        "ci_upper": float(primary_bootstrap["mean_ci_upper"]),
        "interpretation": (
            "Negative values favor Advanced traditional; positive values favor Robust hybrid."
        ),
    },
])

statistical_summary.to_csv(
    RESULTS_DIR / "11_manuscript_statistical_summary.csv",
    index=False,
)

display(statistical_summary)

## Output summary

In [ ]:
output_summary = pd.DataFrame([
    {
        "category": "Statistical results",
        "location": "results/statistical_analysis/",
        "description": "Friedman, Wilcoxon-Holm, bootstrap, Diebold-Mariano and task summaries",
    },
    {
        "category": "Figure source data",
        "location": "results/statistical_analysis/figure_03_data.csv; figure_04_data.csv; figure_05_data.csv",
        "description": "Source data used to reproduce Figures 3, 4 and 5",
    },
    {
        "category": "Main figures",
        "location": "figures/article/",
        "description": "Figures 3, 4 and 5 in PNG and PDF formats",
    },
    {
        "category": "Supplementary figure",
        "location": "figures/article/S01_average_family_rank.*",
        "description": "Average RMSE rank across model families",
    },
])

output_summary.to_csv(
    RESULTS_DIR / "12_output_summary.csv",
    index=False,
)

display(output_summary)

print("Statistical analysis completed.")
print("Results: results/statistical_analysis/")
print("Figures: figures/article/")

## Completion

A complete execution produces:

- the Friedman omnibus comparison;
- pairwise Wilcoxon signed-rank tests with Holm correction;
- paired bootstrap confidence intervals;
- task-level Diebold–Mariano comparisons involving the robust hybrid family;
- Figures 3, 4 and 5 in PNG and PDF formats;
- source-data CSV files for each manuscript figure;
- a concise manuscript-ready statistical summary.

All quantities are calculated from the outputs generated by notebooks 01–09 during the current reproducible workflow.